**IMPORTANT:**

You need to run the FastAPI app for connecting to the endpoints!

In [ ]:
from pydantic_settings import BaseSettings

class Settings(BaseSettings):
    DB_USER: str
    DB_PASSWORD: str
    DB_HOST: str = "localhost"
    DB_PORT: int = 5432
    DB_NAME: str

    ADMIN_USERNAME: str
    ADMIN_PASSWORD: str

    # JWT / Security settings
    SECRET_KEY: str
    ALGORITHM: str = "HS256"
    ACCESS_TOKEN_EXPIRE_MINUTES: int = 30  # Default 30 minutes expiry

    SEARCH_SECRET_KEY: str
    CURRENT_SEARCH_API_VERSION: str = "v1.0.0" # Used to expire tokens when API changes

    # Filter settings
    MAX_ITEMS: int = 5  # Maximum number of items to keep search token size manageable
    MAX_TOKEN_LENGTH: int = 1400  # Safe for URLs

    class Config:
        env_file = "../phsar/.env"  # Tell Pydantic to load from .env

settings = Settings()

In [ ]:
import httpx

BASE_URL = "http://localhost:8000"

async def test_login(username: str, password: str):
    async with httpx.AsyncClient() as client:
        login_payload = {
            "username": username,
            "password": password,
        }
        login_response = await client.post(
            f"{BASE_URL}/auth/login",
            data=login_payload,  # Using form data for OAuth2PasswordRequestForm
            headers={"Content-Type": "application/x-www-form-urlencoded"},
        )
        print("Login status:", login_response.status_code)

        if login_response.status_code != 200:
            print("Login failed:", login_response.text)
        else:
            token_data = login_response.json()
            print("Access token:", token_data["access_token"])
            return token_data

# Run the test
token_data = await test_login(settings.ADMIN_USERNAME, settings.ADMIN_PASSWORD)
token = token_data["access_token"]

# Test `search/mal` and `save/search-results` endpoints

In [ ]:
import httpx

BASE_URL = "http://localhost:8000"

async def test_search_and_save(query: str, token: str = None):
    # Modify the read time for bigger anime franchises (currently 2 min)
    timeout = httpx.Timeout(connect=5.0, read=120.0, write=10.0, pool=5.0)

    headers = {}
    if token:
        headers["Authorization"] = f"Bearer {token}"

    async with httpx.AsyncClient(timeout=timeout) as client:
        # 1. Call the /search/mal endpoint
        search_response = await client.get(f"{BASE_URL}/search/mal", params={"query": query}, headers=headers)
        print("Search status:", search_response.status_code)

        if search_response.status_code != 200:
            print("Search failed:", search_response.text)
            return

        search_results = search_response.json()
        print(f"Got {len(search_results)} results")

        # 2. Call the /save/search-results endpoint with the search results
        save_response = await client.post(
            f"{BASE_URL}/save/search-results",
            json=search_results,
            headers=headers
        )
        print("Save status:", save_response.status_code)
        print("Save response:", save_response)

# Run the tests
await test_search_and_save("My Hero", token=token)

# Test `search/media` endpoint

In [ ]:
import httpx

BASE_URL = "http://localhost:8000"

async def test_search_media(
    query: str,
    relation_type: str = None,
    genre_name: list[str] = None,
    search_type: str = None,
    token: str = None
):
    # Timeout settings for larger or slower responses
    timeout = httpx.Timeout(connect=5.0, read=20.0, write=10.0, pool=5.0)

    params = {"query": query}
    if relation_type:
        params["relation_type"] = relation_type
    if genre_name:
        params["genre_name"] = genre_name
    if search_type:
        params["search_type"] = search_type

    headers = {}
    if token:
        headers["Authorization"] = f"Bearer {token}"

    async with httpx.AsyncClient(timeout=timeout) as client:
        response = await client.get(f"{BASE_URL}/search/media", params=params, headers=headers)
        print("Search/media status:", response.status_code)

        if response.status_code != 200:
            print("Search/media failed:", response.text)
            return

        results = response.json()
        print(f"Search/media returned {len(results)} result(s)")
        print("Sample result:", results[0] if results else "No results")

# Run the test
await test_search_media("My Hero", relation_type="main", genre_name=["Action", "School"], search_type="title", token=token) # Search by title
await test_search_media("Trainings arc", genre_name=["Action", "School"], search_type="description", token=token) # Search by description
await test_search_media("", relation_type="main", genre_name=["Action", "School"], token=token) # Search with empty query

# Test `seed/media` endpoint

**Recommendation:** Comment out some of the animes in `phsar/app/seeders/media_seeder.py` to avoid a script running multiple hours!

*(Or adjust the read timeout)*

In [ ]:
import httpx

BASE_URL = "http://localhost:8000"

async def trigger_media_seeder(token: str = None):
    # Timeout settings
    timeout = httpx.Timeout(connect=5.0, read=60.0*60, write=10.0, pool=5.0)

    headers = {}
    if token:
        headers["Authorization"] = f"Bearer {token}"

    async with httpx.AsyncClient(timeout=timeout) as client:
        response = await client.post(f"{BASE_URL}/seed/media", headers=headers)
        print("Seeder status:", response.status_code)

        if response.status_code != 200:
            print("Seeder failed:", response.text)
        else:
            print("Seeder response:", response.json())

# Run the seeder
await trigger_media_seeder(token=token)

# Test `filters/` enpoint

In [ ]:
import httpx

BASE_URL = "http://localhost:8000"

async def test_filters(token: str = None):
    # Timeout settings
    timeout = httpx.Timeout(connect=5.0, read=20.0, write=10.0, pool=5.0)

    headers = {}
    if token:
        headers["Authorization"] = f"Bearer {token}"

    async with httpx.AsyncClient(timeout=timeout) as client:
        response = await client.get(f"{BASE_URL}/filters/options", headers=headers)
        print("Filters status:", response.status_code)

        if response.status_code != 200:
            print("Filters failed:", response.text)
        else:
            print("Filters response:", response.json())

# Get the filter values
await test_filters(token=token)